# Assignment 08 :: Clustering and Dimensionality Reduction

## 1. Data Loading and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score # importing libraries

### Loading the Dataset:

In [ ]:
housing=fetch_california_housing()
X=pd.DataFrame(housing.data,columns=housing.feature_names)
y=pd.Series(housing.target,name="MedHouseValue")
df = X.copy()
df["MedHouseValue"] = y
df.head() # loading dataset then converted into a Pandas DataFrame for easier analysis and manipulation

In [ ]:
print("Number of rows and columns:", df.shape)

### Missing values:

In [ ]:
df.info()
df.describe()
df.isnull().sum()
print("Total missing values:", df.isnull().sum().sum())
print("Number of duplicate rows:", df.duplicated().sum()) # The dataset was checked for missing values and duplicate records

#### Exploratory Data Analysis

In [ ]:
df.hist(figsize=(14, 10),bins=30)
plt.suptitle("Distribution of California Housing Features")
plt.tight_layout()
plt.show()# Feature Distributions

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(),annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Correlation Heatmap")
plt.show() # correlation heatmap

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df,x="MedInc",y="MedHouseValue")
plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.show()

##### Before building the models, the dataset is explored to understand its structure, data types, statistical information, and possible data-quality issues. Histograms are used to understand the distribution of the numerical features. They help identify skewness, unusual values, and differences in the feature ranges. A correlation heatmap is used to understand the relationships between the features and the median house value. It helps identify features that may have a stronger relationship with the target variable. The scatter plot helps show whether median income has a relationship with median house value. This can help us understand which features may be useful for predicting house prices.

### Data Preprocessing

In [ ]:
X=df.drop("MedHouseValue", axis=1)
y=df["MedHouseValue"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

##### The target variable is separated from the input features. The input features will be used to predict the median house value. The dataset was divided into 80% training data and 20% testing data. The training data is used to train the models, while the testing data is used to evaluate their performance on unseen data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##### Standardisation was used to bring the features to a similar scale. This is important for models such as Linear Regression and SVR because they can be affected by differences in feature ranges. The scaler was fitted only on the training data to avoid data leakage. Decision Tree, Random Forest, and Gradient Boosting models do not require feature scaling.

## 2. Regression Algorithm Implementation

##### --- Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_prediction = linear_model.predict(X_test_scaled)

##### Linear Regression predicts the target variable by finding a linear relationship between the input features and house prices.It is suitable as a baseline model because it is simple and easy to interpret. However, it may not capture complex, non-linear relationships in the dataset.

##### --- Decision Tree Regressor

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)
tree_prediction = tree_model.predict(X_test)

##### A Decision Tree Regressor makes predictions by splitting the data into smaller groups based on feature values. It is suitable for this dataset because it can capture non-linear relationships between housing features and house prices.

##### --- Random Forest Regressor

In [ ]:
forest_model = RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1)
forest_model.fit(X_train, y_train)
forest_prediction = forest_model.predict(X_test)

##### Random Forest combines multiple decision trees and averages their predictions.This usually produces more stable predictions than a single decision tree.It is suitable for this dataset because it can capture complex relationships and reduce the risk of overfitting.

##### --- Gradient Boosting Regressor

In [ ]:
gradient_model = GradientBoostingRegressor(random_state=42)
gradient_model.fit(X_train, y_train)
gradient_prediction = gradient_model.predict(X_test)

##### Gradient Boosting builds models sequentially. Each new model attempts to reduce the errors made by the previous models. It is suitable for this dataset because it can learn complex patterns and non-linear relationships.

##### --- Support Vector Regressor

In [ ]:
svr_model = SVR()
svr_model.fit(X_train_scaled, y_train)
svr_prediction = svr_model.predict(X_test_scaled)

##### Support Vector Regression attempts to predict values within an acceptable error margin. It is suitable for this dataset because it can model non-linear relationships when an appropriate kernel is used. Feature scaling is important for SVR.

## 3. Model Evaluation and Comparison

In [ ]:
def evaluate_model(model_name, actual_values, predicted_values):
    mse = mean_squared_error(actual_values, predicted_values)
    mae = mean_absolute_error(actual_values, predicted_values)
    r2 = r2_score(actual_values, predicted_values)

    return {
        "Model": model_name,
        "MSE": mse,
        "MAE": mae,
        "R2 Score": r2
    }

In [ ]:
results = []

results.append(
    evaluate_model("Linear Regression", y_test, linear_prediction)
)

results.append(
    evaluate_model("Decision Tree", y_test, tree_prediction)
)

results.append(
    evaluate_model("Random Forest", y_test, forest_prediction)
)

results.append(
    evaluate_model("Gradient Boosting", y_test, gradient_prediction)
)

results.append(
    evaluate_model("SVR", y_test, svr_prediction)
)

results_df = pd.DataFrame(results)

results_df

In [ ]:
best_model = results_df.loc[results_df["R2 Score"].idxmax()]
worst_model = results_df.loc[results_df["R2 Score"].idxmin()]

print("Best Model:")
print(best_model)

print("\nWorst Model:")
print(worst_model)

##### The models are evaluated using Mean Squared Error, Mean Absolute Error, and R-squared score.

##### - MSE: Lower values indicate better performance.
##### - MAE: Lower values indicate better performance.
##### - R²: Higher values indicate better performance.

##### From this dataset, Random Forest is the best model and worst one is Linear Regression

### k-Fold Cross-Validation

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR())
    ])
}

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean CV R2": scores.mean(),
        "Standard Deviation": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df

##### Taking k=5, Five-fold cross-validation is used to obtain a more reliable estimate of model performance. The dataset is divided into five parts. The model is trained on four parts and tested on the remaining part. This process is repeated five times. The average R² score is used to compare the models.

### Hyperparameter Tuning

##### --- Decision Tree Tuning

In [ ]:
tree_parameters = {
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10]
}

tree_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    tree_parameters,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

tree_grid.fit(X_train, y_train)

print("Best Decision Tree parameters:", tree_grid.best_params_)
print("Best CV R2:", tree_grid.best_score_)

##### Hyperparameter tuning is used to find better settings for the machine learning models.
##### The `max_depth` and `min_samples_split` parameters are tuned.
##### - `max_depth` controls the maximum depth of the tree.
##### - `min_samples_split` controls the minimum number of samples needed to split a node.
##### These parameters help control overfitting.

##### --- Random Forest Tuning

In [ ]:
forest_parameters = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

forest_grid = GridSearchCV(
    RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    forest_parameters,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

forest_grid.fit(X_train, y_train)

print("Best Random Forest parameters:", forest_grid.best_params_)
print("Best CV R2:", forest_grid.best_score_)

##### The following parameters are tuned:
##### - `n_estimators`: Number of trees in the forest.
##### - `max_depth`: Maximum depth of each tree.
##### - `min_samples_split`: Minimum number of samples required to split a node.
##### Increasing the number of trees can improve stability, but it may increase
##### training time.

##### --- Gradient Boosting Tuning

In [ ]:
gradient_parameters = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3]
}

gradient_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gradient_parameters,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

gradient_grid.fit(X_train, y_train)

print("Best Gradient Boosting parameters:", gradient_grid.best_params_)
print("Best CV R2:", gradient_grid.best_score_)

##### The following parameters are tuned:
##### - `n_estimators`: Number of boosting stages.
##### - `learning_rate`: Contribution of each tree.
##### - `max_depth`: Complexity of the individual trees.
##### A smaller learning rate may improve generalisation, but it may require more trees.

##### --- SVR Tuning

In [ ]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR())
])

svr_parameters = {
    "model__C": [1, 10, 100],
    "model__gamma": ["scale", "auto"],
    "model__epsilon": [0.1, 0.2]
}

svr_grid = GridSearchCV(
    svr_pipeline,
    svr_parameters,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

svr_grid.fit(X_train, y_train)

print("Best SVR parameters:", svr_grid.best_params_)
print("Best CV R2:", svr_grid.best_score_)

##### The following parameters are tuned:
##### - `C`: Controls the balance between model complexity and training error.
##### - `gamma`: Controls the influence of individual data points.
##### - `epsilon`: Defines the acceptable error margin.
##### A pipeline is used so that scaling is performed correctly within each
##### cross-validation fold.

### Evaluate the Tuned Models

In [ ]:
tuned_models = {
    "Decision Tree": tree_grid.best_estimator_,
    "Random Forest": forest_grid.best_estimator_,
    "Gradient Boosting": gradient_grid.best_estimator_,
    "SVR": svr_grid.best_estimator_
}

In [ ]:
tuned_results = []

for name, model in tuned_models.items():
    prediction = model.predict(X_test)

    tuned_results.append(
        evaluate_model(name, y_test, prediction)
    )

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df

### Compare the Tuned Models

In [ ]:
best_model = tuned_results_df.loc[
    tuned_results_df["R2 Score"].idxmax()
]

worst_model = tuned_results_df.loc[
    tuned_results_df["R2 Score"].idxmin()
]

print("Best-performing model:")
print(best_model)

print("\nWorst-performing model:")
print(worst_model)

In [ ]:
linear_result = evaluate_model(
    "Linear Regression",
    y_test,
    linear_prediction
)

final_results = pd.concat(
    [
        pd.DataFrame([linear_result]),
        tuned_results_df
    ],
    ignore_index=True
)

final_results

## 5. Selecting the Best Regression Model

In [ ]:
best_final_model = final_results.loc[
    final_results["R2 Score"].idxmax()
]

print("Selected best model:")
print(best_final_model)

##### The final model was selected by comparing the MSE, MAE, R² score, and cross-validation results.
##### A good regression model should have:
##### - Low MSE
##### - Low MAE
##### - High R² score
##### - Consistent cross-validation performance


In [29]:
print(results_df)

               Model       MSE       MAE  R2 Score
0  Linear Regression  0.555892  0.533200  0.575788
1      Decision Tree  0.495235  0.454679  0.622076
2      Random Forest  0.255368  0.327543  0.805123
3  Gradient Boosting  0.293997  0.371643  0.775645
4                SVR  0.357004  0.398599  0.727563


In [30]:
# Best model: highest R² score
best_model = results_df.loc[results_df["R2 Score"].idxmax()]

# Worst model: lowest R² score
worst_model = results_df.loc[results_df["R2 Score"].idxmin()]

print("Best Model:")
print(best_model)

print("\nWorst Model:")
print(worst_model)

Best Model:
Model       Random Forest
MSE              0.255368
MAE              0.327543
R2 Score         0.805123
Name: 2, dtype: object

Worst Model:
Model       Linear Regression
MSE                  0.555892
MAE                    0.5332
R2 Score             0.575788
Name: 0, dtype: object


##### The best regression model was Random Forest because it achieved the highest R² score and the lowest prediction errors. The worst model was Linear Regression because it had the lowest R² score and comparatively higher prediction errors. The final model was selected by considering the test-set metrics, cross-validation results, and hyperparameter-tuning performance.